In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import torch
import os
import json
import time
import re
import textwrap
import warnings
import shutil
import pandas as pd

import subprocess
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from sentence_transformers import SentenceTransformer, util, CrossEncoder
#import simple_icd_10_cm as icd
from IPython.display import FileLink, display
#from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline 
import transformers
import huggingface_hub
from huggingface_hub import login
from tqdm import tqdm

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("OPENAI_API_KEY")

from openai import OpenAI

client = OpenAI(api_key=api_key)


class OpenAIChatWrapper:
    def __init__(self, client, model):
        self.client = client
        self.model = model

    def __call__(
        self,
        messages,
        max_new_tokens=4096,
        eos_token_id=None,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=None,
    ):
        completion = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            max_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        text = completion.choices[0].message.content or ""
        return [{"generated_text": list(messages) + [{"role": "assistant", "content": text}]}]

ConnectionError: Connection error trying to communicate with service.

In [5]:
###Audio processing agent which converts adudio to transcript######
# Initialize once

import torch
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    pipeline,
)

# ---------------------------------------------------------------------
# Device configuration
# ---------------------------------------------------------------------
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# ---------------------------------------------------------------------
# Load Whisper model
# ---------------------------------------------------------------------
model_id = "openai/whisper-large-v3-turbo"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)

model.to(device)

# ---------------------------------------------------------------------
# Create ASR pipeline
# ---------------------------------------------------------------------
asr_pipeline = pipeline(
    task="automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    return_timestamps=True,
    torch_dtype=torch_dtype,
    device=device,
)

# ---------------------------------------------------------------------
# Audio-to-Transcript Agent
# ---------------------------------------------------------------------
def audio2transcript_agent(audio_path: str):
    """
    Converts an audio file into transcript with timestamps.

    Args:
        audio_path (str): Path to audio file.

    Returns:
        dict
    """

    result = asr_pipeline(audio_path)

    return {
        "audio_file": audio_path,
        "transcript": result["text"],
        "timestamps": result.get("chunks", []),
    }


# Example
# result = audio2transcript_agent("sample.wav")
# print(result["transcript"])

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (4 outputs left out)

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [10]:
!find /kaggle/input -type f -iname "Swarnima.mp4"

/kaggle/input/datasets/sumanroyiisc/datacollector/Swarnima.mp4


In [11]:
!ffmpeg -y \
    -i "/kaggle/input/datasets/sumanroyiisc/datacollector/Swarnima.mp4" \
    -vn \
    -ac 1 \
    -ar 16000 \
    -c:a pcm_s16le \
    "/kaggle/working/Swarnima.wav"

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [12]:
response = audio2transcript_agent("/kaggle/working/Swarnima.wav")

transcript = response["transcript"]

with open("transcript.txt", "w", encoding="utf-8") as f:
    f.write(transcript)

print("Transcript saved as transcript.txt")
print(transcript)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

Transcript saved as transcript.txt
 नमस्ते माम मेरे नाम डॉक्टर प्रियांची चहान है मैं आपसे कुछ सवाल पूछना चाहती हूँ आपका नाम क्या है स्वाइडे माम और आपकी उमर तीस और आपकी डेट ऑफ बर्थ क्या है तीस पांच नैंटी नैंटी फाइफ और आपने कहां तक पढ़ाई गरी है मैंने अमबीबियस किया हुआ है और आप भी आप क्या जॉब कर रहे हैं मैं पी जी जी यार हूं कम्यूटी मेडिसन देपार्टमेंट में आपकी शादी हो गई है हाजी और आपका एड्रेस क्या है नौएडा आपके पती का नाम क्या है राजवाश ने आपना फोन नमर बता देंगे मेरे को 9634 551537 आपकी महीने की फैमिली इंकम कितनी है एक महीने में तू लैक्स आपने कोई हेल्थ इंशॉरेंस जैसे की गवर्मेंट का चल रहा है आयुश्मान एस आईसी इसी में आपने आप अवेल कर रहे हैं कोई नहीं आपका वजन कितना होगा और हाइट पाइव तू आपको कोई लंबे बिमारी रही है जैसे कि डाइबेटीज हाइपर बीपी नहीं से कोई बिमारी आपने कभी किसी तरीके का कैंसर की जांच कराई है जैसे कि ब्रेस्ट कैंसर औरल कैंसर किस कैंसर की सर्वाइकल आपकी फैमिली में कोई है जिसे कभी कैंसर रहा हो नहीं आपने कभी शराब या फिर तंबाको का सेवन किया है नहीं आप किस तरह का खाना खाती है वेजिटेरियन 

In [ ]:
import json

# ------------------------------------------------------------------
# Diarization Agent
# ------------------------------------------------------------------

pipeline_8b_instruct = OpenAIChatWrapper(client, "gpt-4.1-mini")
terminators = None
pipeline_llm = pipeline_8b_instruct

def diarization_agent(
    transcript: str,
    llm_pipeline,
    terminators,
    max_new_tokens: int = 4096,
    temperature: float = 0.2,
    top_p: float = 0.9,
):
    """
    Assign Doctor/Patient speaker labels to an ASR transcript.

    Parameters
    ----------
    transcript : str
        Raw ASR transcript without speaker labels.

    llm_pipeline :
        HuggingFace text-generation/chat pipeline.

    terminators :
        EOS token ids for the model.

    Returns
    -------
    str
        Diarized transcript with Doctor/Patient labels.

    Raises
    ------
    RuntimeError
        If the LLM output cannot be parsed.
    """

    system_prompt = (
        "You are an advanced AI model trained to analyze and structure "
        "medical conversations between a doctor and a patient."
    )

    user_prompt_template = """
You are given a two party conversation transcript between a Doctor and a Patient that was generated by an ASR engine. The transcript does not contain speaker labels.

Your task is to correctly assign speaker roles as Doctor or Patient based on the context.

Guidelines

Doctor:
- Asks diagnostic questions
- Collects medical history
- Gives medical advice
- Uses structured questioning

Patient:
- Describes symptoms
- Answers doctor's questions
- Shares personal medical history

Output Instructions

Return ONLY the following JSON:

[
    {
        "text": "<speaker labelled transcript>"
    }
]

Do not change the original text.
Do not remove any sentence.
Only insert speaker labels.

Medical Conversation

%s

Output:
"""

    sample_input = (
        "what brought you in today? Sure. I'm just having a lot of chest pain, "
        "and so I thought I should get it checked out. "
        "Okay. And before we start, could you remind me of your gender and age? "
        "Sure. I'm 39. I'm a male. "
        "Okay. And so when did this chest pain start?"
    )

    sample_output = """
[
    {
        "text":"Doctor: What brought you in today?
Patient: Sure. I'm just having a lot of chest pain, and so I thought I should get it checked out.
Doctor: Okay. And before we start, could you remind me of your gender and age?
Patient: Sure. I'm 39. I'm a male.
Doctor: Okay. And so when did this chest pain start?"
    }
]
"""

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt_template % sample_input,
        },
        {
            "role": "assistant",
            "content": sample_output,
        },
        {
            "role": "user",
            "content": user_prompt_template % transcript,
        },
    ]

    outputs = llm_pipeline(
        messages,
        max_new_tokens=max_new_tokens,
        eos_token_id=terminators,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
    )

    response = outputs[0]["generated_text"][-1]["content"]

    try:
        parsed = json.loads(response, strict=False)
        return parsed[0]["text"]

    except Exception as e:
        raise RuntimeError(
            f"Unable to parse diarization response.\n\n"
            f"LLM Output:\n{response}\n\n"
            f"Error: {e}"
        )

In [ ]:
# Read transcript from file
with open("transcript.txt", "r", encoding="utf-8") as f:
    transcript = f.read()
    
diarized_text = diarization_agent(
    transcript=transcript,
    llm_pipeline=pipeline_8b_instruct,
    terminators=terminators,
)

print(diarized_text)